# EDA (exploratory data analysis)

Notebook 01 generates the data. This one is about **understanding** it before we cluster
it: what the two features look like, how the clusters differ, and what the extra points
actually do to the statistics.

We load the saved file, so we are looking at exactly the same numbers as every other
notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

d = np.load("data/blobs.npz")

for k in d.files:
    print(f"{k:10} {d[k].shape} {d[k].dtype}")

X_clean, y_clean = d["X_clean"], d["y_clean"]
X_out, y_out = d["X_out"], d["y_out"]

### A first summary

`describe()` gives the range, the centre and the spread of each feature. These are the
numbers a clustering method works with — and for k-means the centre and the spread are
literally what it optimises.

In [ ]:
df = pd.DataFrame(X_clean, columns=["feature 1", "feature 2"])
df.describe().round(2)

### Each feature individually

A histogram of one feature ignores the clusters completely.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col in zip(axes, df.columns):
    ax.hist(df[col], bins=30, color="0.6", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("value")

axes[0].set_ylabel("count")
plt.tight_layout()
plt.show()

### Now split by cluster

This is where the interesting part shows. The clusters are **not equally spread out**: we
passed `cluster_std=[0.6, 1.2, 0.6, 1.8]` in notebook 01, so the widest cluster has a
standard deviation roughly **three times** that of the tightest ones.

k-means assumes clusters of similar spread. Keep that in mind.

In [ ]:
summary = (
    df.assign(cluster=y_clean)
      .groupby("cluster")
      .agg(n=("feature 1", "size"),
           mean_1=("feature 1", "mean"), std_1=("feature 1", "std"),
           mean_2=("feature 2", "mean"), std_2=("feature 2", "std"))
      .round(2)
)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col in zip(axes, df.columns):
    df.assign(cluster=y_clean).boxplot(column=col, by="cluster", ax=ax, grid=False)
    ax.set_title(f"{col} by cluster")
    ax.set_xlabel("cluster")

plt.suptitle("")
plt.tight_layout()
plt.show()

### Are the two features related?

Compute the correlation over all points.

In [ ]:
print("correlation over all points")
print(df.corr().round(3))
print()
print("correlation within each cluster")
for k in sorted(np.unique(y_clean)):
    r = df[y_clean == k].corr().iloc[0, 1]
    print(f"  cluster {k}:  r = {r:+.3f}")

### With the outliers

15 points out of 315; under 5%. Look at the effect on the statistics.

In [ ]:
comparison = pd.DataFrame({
    "clean":         [X_clean[:, 0].mean(), X_clean[:, 0].std(),
                      X_clean[:, 1].mean(), X_clean[:, 1].std()],
    "with outliers": [X_out[:, 0].mean(),   X_out[:, 0].std(),
                      X_out[:, 1].mean(),   X_out[:, 1].std()],
}, index=["mean f1", "std f1", "mean f2", "std f2"]).round(2)

print(comparison, "\n")

centre = X_clean.mean(axis=0)
dist = np.linalg.norm(X_out - centre, axis=1)
print("distance to the overall centre of the clean data:")
print(f"  furthest of the 300 original points : {dist[:len(X_clean)].max():6.2f}")
print(f"  closest of the 15 added points      : {dist[len(X_clean):].min():6.2f}")

**The means barely move, but the standard deviations inflate%**
For k-means this is the relevant damage: the objective is
built on squared distances to a *mean*, so a handful of far-away points pulls the centre
and stretches the cluster.